# பாடம் 11 - முகவர்-முகவர் (A2A) புரอตோக்கால்


## அமைப்பு


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## What is the A2A Protocol?

The **Agent-to-Agent (A2A) protocol** is an open standard that enables AI agents to communicate,
discover each other, and collaborate — even when they are built on different frameworks or hosted
by different services.

Key concepts:

- **Discovery** – Agents publish an *Agent Card* that describes their capabilities, making it
  easy for other agents (or orchestrators) to find the right specialist for a task.
- **Message Passing** – Agents exchange structured messages through a common protocol, so a
  request from one agent can be understood and fulfilled by another regardless of its internal
  implementation.
- **Task Lifecycle** – A2A defines states such as *submitted*, *working*, *completed*, and
  *failed*, giving the orchestrator full visibility into how a delegated task is progressing.

In this lesson we simulate A2A-style collaboration by wiring three specialized travel agents
into a workflow where each agent contributes its expertise and passes results to the next.


## சிறப்பு பயண முகவர்களை உருவாக்குதல்


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## பணிப்பாளர் ஒருங்கிணைப்புடன் பல முகவர் ஒத்துழைப்பு

நாங்கள் மூன்று முகவர்களை A2A செய்திகள் பரிமாற்றத்தை பிரதிபலிக்கும் தொடர்ச்சியான பணிப் பணியில் இணைக்கிறோம்:

1. **CurrencyExchangeAgent** பயனர் கோரிக்கையை பெற்று நாணய வழிகாட்டுதலை உருவாக்குகிறான்.
2. **ActivityPlannerAgent** சோம்பல் நிறைந்த சூழலைப் பெற்று செயல்பாட்டு பரிந்துரைகளை சேர்க்கின்றான்.
3. **TravelManagerAgent** இரு உள்ளீடுகளை இறுதிச் பயண சுருக்கமாக சங்கீனிக்கின்றான்.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## உற்பத்தியில் A2A-வை புரிந்துகொள்வது

உற்பத்தி சூழலில் A2A நெறிமுறை சக்திவாய்ந்த சேவை-கடந்த அனுபவங்களைக் திறக்கிறது:

| திறன் | விளக்கம் |
|---|---|
| **கடந்து-வடிவமைப்பு இடையாட்டம்** | ஒன்றரை வடிவமைப்பில் கட்டப்பட்ட ஒரு முகவர் எந்தவொரு மற்ற A2A-ஐ பின்பற்றும் வடிவமைப்பிலும் கட்டப்பட்ட முகவருக்கு பணிகளை ஒப்படைக்க முடியும், இது உண்மையான கடந்து-நிறுவன இடையாட்டத்தை சாத்தியமாக்குகிறது. |
| **சேவை எல்லைகள்** | முகவர்கள் தனித்தனியான மைக்ரோசேவைகள், மேக பிராந்தியங்கள் அல்லது வேறுபட்ட நிறுவனங்களில் வாழக்கூடியபோதும் நிலைத்துடனான ஒத்துழைப்பை மேற்கொள்ளலாம். |
| **ஊர்வழி கண்டறிதல்** | ஒருங்கிணைப்பாளர் ஒரு முகவர் அட்டை பதிவுப் பட்டியலை செயல்பாட்டின் போது கேள்வி எழுப்பி ஒரு குறிப்பிட்ட துணைபணி சிறப்பாளரை கண்டுபிடிக்க முடியும். |
| **நடப்பு மற்றும் பால்சூழல் அறிவிப்புகள்** | A2A நேரடி முன்னேற்ற புதுப்பிப்புகளுக்காக Server-Sent Events (SSE) ஐ மற்றும் நீண்டநேர பணிகளுக்கான பால்சூழல் அறிவிப்புகளை ஆதரிக்கிறது. |

மேல் நாம் கட்டிய வேலைப்பயணம் இந்த மாதிரியான ஒரு எளிமையாக்கப்பட்ட, செயல்முறை உள்ளமைவு பதிப்பாகும். உண்மையான
தளவமைப்பில் ஒவ்வொரு முகவரும் HTTP இறுதிப்புள்ளியை வெளிப்படுத்தி, முகவர் அட்டை ஒன்றை வெளியிட்டு,
A2A JSON-RPC நெறிமுறையினால் தொடர்பு கொள்ளும்.


## சுருக்கம்

இந்த பாடத்தில் நீங்கள் கற்றுக்கொண்டது:

1. **A2A நெறிமுறை என்றால் என்ன என்பது** — முகவர்-முகவர் கண்டுபிடிப்பு, செய்தியறிதல்,
   மற்றும் பணி மேலாண்விற்கான ஓர் திறந்த தரநிலை.
2. **திறமையான முகவர்களை எப்படி உருவாக்குவது** — ஸ்திரி பரிவர்த்தனை முகவர், செயல்பாட்டு திட்டமிடும் முகவர்,
   மற்றும் பயண மேலாளர் ஒருங்கிணைப்பாளர்.
3. **முகவர்களை பணிவகையில் இணைப்பது எப்படி** — முகவர்களுக்கிடையில் தொடர் செய்தி பரிமாற்றத்தைக்
   மாதிரிவரையிட `WorkflowBuilder` பயன்படுத்துதல்.
4. **A2A உற்பத்தியில் எப்படி செயல்படுகிறது** — இயக்கக்குழு, சேவை சாரா ஒத்துழைப்பு
   மற்றும் மாறுமாறு கண்டுபிடிப்பு உடன் கலைஞர் புதுப்பிப்புகள் அமையுதல்.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
